In [ ]:
import os
import urllib.request
import zipfile
import shutil
import scanpy as sc
import numpy as np
import pandas as pd

In [3]:
def download_and_extract_scrna():
    # Set up the download path
    download_path = os.path.expanduser("../data")
    os.makedirs(download_path, exist_ok=True)
    
    # Download URL
    url = "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/v6n743h5ng-1.zip"
    zip_file = os.path.join(download_path, "v6n743h5ng-1.zip")
    
    print(f"Downloading from {url}...")
    urllib.request.urlretrieve(url, zip_file)
    print(f"Downloaded to {zip_file}")
    
    # Extract the outer zip file to a temp directory
    temp_extract = os.path.join(download_path, "temp_extract")
    os.makedirs(temp_extract, exist_ok=True)
    
    print("Extracting outer zip file...")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(temp_extract)
    
    # Find and extract the scRNAseq.zip file
    scrna_zip = os.path.join(temp_extract, "scRNAseq.zip")
    if os.path.exists(scrna_zip):
        print("Extracting scRNAseq.zip...")
        with zipfile.ZipFile(scrna_zip, 'r') as zip_ref:
            zip_ref.extractall(temp_extract)
    
    # Move contents of scRNAseq folder to the main directory
    scrna_folder = os.path.join(temp_extract, "scRNAseq")
    if os.path.exists(scrna_folder):
        print("Moving scRNAseq contents to main directory...")
        for item in os.listdir(scrna_folder):
            src = os.path.join(scrna_folder, item)
            dst = os.path.join(download_path, item)
            if not os.path.exists(dst):
                shutil.move(src, dst)
            print(f"  Moved: {item}")
    
    # Clean up temp directory and zip file
    shutil.rmtree(temp_extract, ignore_errors=True)
    os.remove(zip_file)
    print("Cleaned up temporary files")
    
    print(f"\nDone! scRNAseq data is in: {download_path}")
    print("\nFinal contents:")
    for item in os.listdir(download_path):
        print(f"  - {item}")

# Run the function
download_and_extract_scrna()

Downloaded to ../data/v6n743h5ng-1.zip
Extracting outer zip file...
Extracting scRNAseq.zip...
Moving scRNAseq contents to main directory...
  Moved: T0_1A
  Moved: T2_3B
  Moved: T4_5C
  Moved: T6_7D
  Moved: T8_9E
Cleaned up temporary files

Done! scRNAseq data is in: ../data

Final contents:
  - T0_1A
  - T2_3B
  - T4_5C
  - T6_7D
  - T8_9E
  - embryoid_body_gene.npy
  - embryoid_body_preprocessed.npy
  - embryoid_body_timepoint.npy


In [4]:
data_path = os.path.expanduser("../data")

samples = ['T0_1A', 'T2_3B', 'T4_5C', 'T6_7D', 'T8_9E']
labels = ['Day 00-03', 'Day 06-09', 'Day 12-15', 'Day 18-21', 'Day 24-27']

adatas = []
for sample, label in zip(samples, labels):
    adata = sc.read_10x_mtx(os.path.join(data_path, sample), var_names='gene_symbols', make_unique=True,cache=True)
    adata.obs['timepoint'] = label
    adatas.append(adata)

# Concatenate all samples
adata = sc.concat(adatas, merge="same")
adata.obs_names_make_unique()
print(f"Loaded {adata.n_obs} cells and {adata.n_vars} genes")

Loaded 31161 cells and 33694 genes


/home/cpsc4844_tl855/project_cpsc4844/cpsc4844_tl855/uPHATE/.venv/lib64/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [5]:
# Calculate QC metrics
## Here we are also computing quality control specific for Mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [6]:
# Filter cells per sample based on library size percentiles
cells_to_keep = []
min_percentile = 20 # Filter cells lower than the 20th percentile
max_percentile = 80 # Filter cells higher than the 80th percentile

for sample, label in zip(samples, labels):
    # Get cells for this sample
    sample_mask = adata.obs['timepoint'] == label
    sample_counts = adata.obs.loc[sample_mask, 'total_counts']
    
    # Calculate thresholds for this sample
    threshold_min = np.percentile(sample_counts, min_percentile)
    threshold_max = np.percentile(sample_counts, max_percentile)
    
    # Identify cells to keep for this sample
    keep_mask = (sample_counts >= threshold_min) & (sample_counts <= threshold_max)
    cells_to_keep.extend(sample_counts[keep_mask].index.tolist())
    
    print(f"{label}: keeping {keep_mask.sum()}/{len(sample_counts)} cells "
          f"(range: {threshold_min:.0f} - {threshold_max:.0f})")
    
# Filter the AnnData object
adata = adata[cells_to_keep, :].copy()
print(f"\nTotal cells after filtering: {adata.n_obs}")

Day 00-03: keeping 2789/4649 cells (range: 4406 - 9874)
Day 06-09: keeping 4425/7377 cells (range: 3194 - 6486)
Day 12-15: keeping 3747/6245 cells (range: 2638 - 5674)
Day 18-21: keeping 3939/6563 cells (range: 2721 - 5672)
Day 24-27: keeping 3796/6327 cells (range: 1935 - 4256)

Total cells after filtering: 18696


In [7]:
# Eliminate genes that are expressed in few cells
min_cells = 10
sc.pp.filter_genes(adata, min_cells=min_cells)

# Normalize to median library size
sc.pp.normalize_total(adata, target_sum=np.median(adata.obs['total_counts']))

# Get mitochondrial percentages
mito_pct = adata.obs['pct_counts_mt']
mito_percentile = 90
mito_threshold = np.percentile(mito_pct, mito_percentile)
# Filter cells with high mitochondrial content
adata = adata[adata.obs['pct_counts_mt'] < mito_threshold].copy()

# Square root transform (alternative to log transform, stable at zero)
# sc.pp.log1p(adata)  # You can also use log1p if preferred
adata.X = np.sqrt(adata.X) # We use 

In [8]:
# Square root transform (alternative to log transform, stable at zero)
# sc.pp.log1p(adata)  # You can also use log1p if preferred
adata.X = np.sqrt(adata.X)

In [ ]:
import time

start = time.time()
sc.tl.pca(adata, n_comps=500)  # Computes PCA and stores in adata.obsm['X_pca']
end = time.time()
print(f"PCA done in {end-start:.1f} seconds")

In [26]:
np.save("../data/embryoid_body_preprocessed_pca.npy", adata.obsm["X_pca"])
np.save("../data/embryoid_body_preprocessed.npy", adata.X)
np.save("../data/embryoid_body_timepoint.npy", pd.factorize(adata.obs["timepoint"])[0])
np.save("../data/embryoid_body_gene.npy", adata.var.index)